In [0]:
catalog = "meu_catalog"

bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

print(f"Bronze: {bronze_schema}")
print(f"Silver: {silver_schema}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

#As funções abaixo centralizam a limpeza de números e textos.

def normalizar_texto(coluna):
    """Trim + colapsa espaços em branco. Não altera o conteúdo semântico."""
    return F.regexp_replace(
        F.trim(F.col(coluna).cast("string")),
        r"\s+",
        " "
    )


def numero_limpo(coluna):
    """
    Normaliza números monetários/quantitativos sem usar try_cast.

    Suporta:
    - moedas/símbolo: USD, $, €, £ e R$
    - espaços
    - decimal BR: 1.234,56 / 1234,56
    - decimal US: 1,234.56
    - separador de milhar: 1,234 / 1.234
    - sufixos K/M: 10.0K -> 10000; 9.9M -> 9900000

    O resultado só é convertido quando o texto final tem formato numérico válido.
    Portanto, a função não depende de ANSI/try_cast e não tenta converter lixo.
    """
    c = F.trim(F.col(coluna).cast("string"))

    c = F.when(
        c.isNull()
        | F.lower(c).isin(
            "unknown", "não informado", "nao informado",
            "null", "none", "n/a", "na", "nan", ""
        ),
        None
    ).otherwise(c)

    # Remove a moeda explícita "USD" e símbolos monetários.
    c = F.regexp_replace(c, r"(?i)usd", "")
    c = F.regexp_replace(c, r"(?i)^r\$?", "")
    c = F.regexp_replace(c, r"[$€£]", "")
    c = F.regexp_replace(c, r"\s+", "")

    # Guarda o multiplicador dos sufixos abreviados e remove o sufixo.
    fator = (
        F.when(c.rlike(r"[kK]$"), F.lit(1000))
         .when(c.rlike(r"[mM]$"), F.lit(1000000))
         .otherwise(F.lit(1))
    )
    c = F.regexp_replace(c, r"[kKmM]$", "")

    # 1.234,56 -> 1234.56
    c = F.when(
        c.rlike(r"^-?\d{1,3}(\.\d{3})+,\d+$"),
        F.regexp_replace(
            F.regexp_replace(c, r"\.", ""),
            ",",
            "."
        )
    ).otherwise(c)

    # 1,234.56 -> 1234.56
    c = F.when(
        c.rlike(r"^-?\d{1,3}(,\d{3})+\.\d+$"),
        F.regexp_replace(c, ",", "")
    ).otherwise(c)

    # 1,234 -> 1234 (milhar) / 12,5 -> 12.5 (decimal).
    c = F.when(
        c.rlike(r"^-?\d{1,3}(,\d{3})+$"),
        F.regexp_replace(c, ",", "")
    ).when(
        c.rlike(r"^-?\d+,\d+$"),
        F.regexp_replace(c, ",", ".")
    ).otherwise(c)

    # Só faz cast depois de validar o formato numérico.
    valido = c.rlike(r"^-?\d+(\.\d+)?$")

    return (
        F.when(
            valido,
            c.cast("decimal(38,10)") * fator
        )
        .otherwise(F.lit(None).cast("decimal(38,10)"))
    )


def inteiro_seguro(coluna):
    """Converte somente números inteiros; valores fracionários viram NULL."""
    n = numero_limpo(coluna).cast("double")
    return (
        F.when(
            n.isNotNull() & (n == F.floor(n)),
            n.cast("int")
        )
        .otherwise(F.lit(None).cast("int"))
    )


def data_segura(coluna, formato, regex):
    """Tenta converter a data apenas quando o padrão textual é compatível."""
    c = F.trim(F.col(coluna).cast("string"))
    return F.when(
        c.rlike(regex),
        F.to_date(c, formato)
    ).otherwise(F.lit(None).cast("date"))


def completude(colunas):
    """Pontua a quantidade de campos preenchidos para desempate determinístico."""
    score = F.lit(0)
    for coluna in colunas:
        preenchido = (
            F.col(coluna).cast("string").isNotNull()
            & (F.trim(F.col(coluna).cast("string")) != "")
        )
        score = score + F.when(preenchido, 1).otherwise(0)
    return score


def hash_deterministico(colunas):
    """Hash estável do conteúdo usado como último critério de desempate."""
    valores = [
        F.coalesce(F.col(coluna).cast("string"), F.lit(""))
        for coluna in colunas
    ]
    return F.sha2(F.concat_ws("\u001f", *valores), 256)

In [0]:
'''''
Regras:
- status normalizado antes da tradução;
- status inválido → `Não Informado`;
- unicidade por filme, mantendo a ingestão mais recente;
- datas testadas em múltiplos formatos;
- `ano_lancamento` derivado da data.
'''''

# Proteção para execução isolada desta célula.
# Em uma execução normal com "Run All", a célula de configuração (2) já define essas variáveis.
if "bronze_schema" not in globals():
    catalog = "meu_catalog"
    bronze_schema = f"{catalog}.bronze"
    silver_schema = f"{catalog}.silver"

df_info_raw = spark.table(f"{bronze_schema}.tb_movies_info")

# Desempate determinístico:
# 1) ingestion_datetime mais recente;
# 2) maior completude;
# 3) hash estável do conteúdo.
info_cols = [
    "title", "original_title", "release_date", "runtime",
    "original_language", "status", "overview", "tagline"
]

w_info = Window.partitionBy("id").orderBy(
    F.col("ingestion_datetime").desc_nulls_last(),
    F.col("_completude").desc(),
    F.col("_hash").asc()
)

status = F.regexp_replace(
    F.lower(F.trim(F.col("status"))),
    r"[-_]+",
    " "
)
status = F.regexp_replace(status, r"\s+", " ")

df_info = (
    df_info_raw
    .withColumn("_completude", completude(info_cols))
    .withColumn("_hash", hash_deterministico(info_cols))
    .withColumn("_status", status)
    .withColumn(
        "_status_pt",
        F.when(F.col("_status") == "released", "Lançado")
         .when(F.col("_status") == "post production", "Pós-Produção")
         .when(F.col("_status") == "in production", "Em Produção")
         .when(F.col("_status") == "planned", "Planejado")
         .when(F.col("_status") == "rumored", "Rumores")
         .when(F.col("_status") == "canceled", "Cancelado")
         .otherwise("Não Informado")
    )
    .withColumn("_rn", F.row_number().over(w_info))
    .filter(F.col("_rn") == 1)
)

data_lancamento = F.coalesce(
    F.expr("try_to_date(release_date, 'yyyy-MM-dd')"),
    F.expr("try_to_date(release_date, 'dd/MM/yyyy')"),
    F.expr("try_to_date(release_date, 'MM/dd/yyyy')"),
    F.expr("try_to_date(release_date, 'dd-MM-yyyy')"),
    F.expr("try_to_date(release_date, 'MM-dd-yyyy')"),
    F.expr("try_to_date(release_date, 'yyyy/MM/dd')")
)

df_info = (
    df_info.withColumn("data_lancamento", data_lancamento)
           .withColumn("ano_lancamento", F.year("data_lancamento"))
           .select(
               F.col("id").cast("string").alias("id_filme"),
               normalizar_texto("title").cast("string").alias("titulo"),
               normalizar_texto("original_title").cast("string").alias("titulo_original"),
               F.col("data_lancamento").cast("date"),
               F.col("ano_lancamento").cast("int"),
               inteiro_seguro("runtime").alias("duracao_minutos"),
               normalizar_texto("original_language").cast("string").alias("idioma_original"),
               F.col("_status_pt").cast("string").alias("status_filme"),
               normalizar_texto("overview").cast("string").alias("sinopse"),
               normalizar_texto("tagline").cast("string").alias("frase_divulgacao")
           )
)

df_info.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{silver_schema}.tb_info_filmes")

print(f"{silver_schema}.tb_info_filmes criada")
display(df_info.limit(10))

In [0]:
df = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

df_cot = (
    df.select(
        F.to_timestamp("dataHoraCotacao").alias("data_hora_cotacao"),
        numero_limpo("cotacaoCompra").cast("decimal(18,6)").alias("cotacao_compra")
    )
    .filter(F.col("data_hora_cotacao").isNotNull())
    .filter(F.col("cotacao_compra").isNotNull())
    .withColumn("data_cotacao", F.to_date("data_hora_cotacao"))
)

# Uma cotação por dia: a última retornada pelo BCB naquele dia.
w_dia = Window.partitionBy("data_cotacao").orderBy(
    F.col("data_hora_cotacao").desc()
)

df_diaria = (
    df_cot
    .withColumn("_rn", F.row_number().over(w_dia))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

limites = df_diaria.select(
    F.min("data_cotacao").alias("data_min"),
    F.max("data_cotacao").alias("data_max")
).first()

if limites["data_min"] is None:
    raise ValueError("bronze.tb_cotacao_dolar não possui cotações válidas.")

df_datas = spark.sql(f"""
    SELECT explode(sequence(
        to_date('{limites["data_min"]}'),
        to_date('{limites["data_max"]}'),
        interval 1 day
    )) AS data_cotacao
""")

df_cot_silver = (
    df_datas
    .join(
        df_diaria.select(
            "data_cotacao",
            "data_hora_cotacao",
            "cotacao_compra"
        ),
        "data_cotacao",
        "left"
    )
)

w_ffill = Window.orderBy("data_cotacao").rowsBetween(
    Window.unboundedPreceding,
    Window.currentRow
)

df_cot_silver = (
    df_cot_silver
    .withColumn(
        "cotacao_compra",
        F.last("cotacao_compra", ignorenulls=True).over(w_ffill)
    )
    .select(
        "data_cotacao",
        "data_hora_cotacao",
        F.col("cotacao_compra").cast("decimal(18,6)")
    )
)

df_cot_silver.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{silver_schema}.tb_cotacao_dolar")

display(df_cot_silver)

In [0]:
df = spark.table(f"{bronze_schema}.tb_movies_financials")

# Uma linha por filme antes de calcular valores em USD/BRL.
fin_cols = ["budget", "revenue"]

w_fin = Window.partitionBy("id").orderBy(
    F.col("_completude").desc(),
    F.col("_hash").asc()
)

df_fin_base = (
    df
    .withColumn("_completude", completude(fin_cols))
    .withColumn("_hash", hash_deterministico(fin_cols))
    .withColumn("_rn", F.row_number().over(w_fin))
    .filter(F.col("_rn") == 1)
)

df_fin = (
    df_fin_base
    .select(
        F.col("id").cast("string").alias("id_filme"),
        numero_limpo("budget").cast("decimal(18,2)").alias("_orcamento"),
        numero_limpo("revenue").cast("decimal(18,2)").alias("_receita")
    )
    .withColumn(
        "orcamento_usd",
        F.when(F.col("_orcamento") > 0, F.col("_orcamento"))
    )
    .withColumn(
        "receita_usd",
        F.when(F.col("_receita") > 0, F.col("_receita"))
    )
    .drop("_orcamento", "_receita")
)

taxa = (
    spark.table(f"{silver_schema}.tb_cotacao_dolar")
    .filter(F.col("cotacao_compra").isNotNull())
    .orderBy(F.col("data_cotacao").desc())
    .select(F.col("cotacao_compra").alias("taxa_dolar_brl"))
    .first()
)

if taxa is None:
    raise ValueError("Não foi possível obter uma cotação válida da Silver.")

taxa_dolar = taxa["taxa_dolar_brl"]

df_fin = (
    df_fin
    .withColumn(
        "orcamento_brl",
        F.round(F.col("orcamento_usd") * F.lit(taxa_dolar), 2)
        .cast("decimal(18,2)")
    )
    .withColumn(
        "receita_brl",
        F.round(F.col("receita_usd") * F.lit(taxa_dolar), 2)
        .cast("decimal(18,2)")
    )
    .withColumn(
        "lucro_usd",
        F.when(
            F.col("receita_usd").isNotNull()
            & F.col("orcamento_usd").isNotNull(),
            F.round(
                F.col("receita_usd") - F.col("orcamento_usd"),
                2
            )
        ).cast("decimal(18,2)")
    )
    .withColumn(
        "lucro_brl",
        F.when(
            F.col("receita_brl").isNotNull()
            & F.col("orcamento_brl").isNotNull(),
            F.round(
                F.col("receita_brl") - F.col("orcamento_brl"),
                2
            )
        ).cast("decimal(18,2)")
    )
    .withColumn(
        "margem_lucro_percentual",
        F.when(
            F.col("receita_usd").isNotNull()
            & (F.col("receita_usd") != 0)
            & F.col("lucro_usd").isNotNull(),
            F.round(
                (F.col("lucro_usd") / F.col("receita_usd")) * 100,
                2
            )
        ).cast("decimal(10,2)")
    )
    .select(
        "id_filme", "orcamento_usd", "receita_usd",
        "orcamento_brl", "receita_brl",
        "lucro_usd", "lucro_brl",
        "margem_lucro_percentual"
    )
)

df_fin.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{silver_schema}.tb_financeiro_filmes")

print(f"Cotação aplicada: {taxa_dolar}")
display(df_fin.limit(10))